# Rating Table Optimization (Model 2)

Derives per-corpus gain mappings (`r_table`) for ListNet loss via grid search over label statistics.

## Step 1: Install Dependencies

In [ ]:
!pip install faiss-cpu

## Step 2: Build Training Groups with RRF Fusion

Retrieves top-K candidates per query using weighted RRF over:
- **E5** (multilingual-e5-large): Dense semantic retrieval
- **TF-IDF (word)**: Unigram + bigram sparse retrieval
- **TF-IDF (char)**: Optional character n-gram retrieval

**Outputs**:
- `groups_train_k{K}.jsonl` / `groups_val_k{K}.jsonl`: Train/val groups with labels
- `train_query_uuids.txt` / `val_query_uuids.txt`: Query splits for reproducibility
- Recall@K and upper-bound NDCG@20 metrics per corpus

In [ ]:
# ---------------------- Config ----------------------
import os, json, math, time, random, hashlib, gc
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer

# ---------------------- Corpus keys & aliases ----------------------
CORPUS_KEYS = {
    "mafat_retrieval_wikipedia_corpus": "wiki",
    "mafat_retrieval_kz_corpus":        "kz",
    "mafat_retrieval_knesset_corpus":   "knesset",
}
SLUG_TO_CORPUSKEY = {v: k for k, v in CORPUS_KEYS.items()}

# Accept common misspellings
ALIASES = {"kenesset": "knesset", "wikipedia": "wiki"}

# For teacher file naming
SLUG_TO_TEACHER_CORPUSNAME = {
    "wiki": "wikipedia",
    "kz": "kz",
    "knesset": "knesset",
}

# ---------------------- Paths ----------------------
CORPUS_JSONL   = os.getenv("CORPUS_JSONL", "/content/hsrc_corpus.jsonl")
TRAIN_JSONL    = os.getenv("TRAIN_JSONL",  "/content/hsrc_train_augmented.jsonl")

# NEW: default to a fresh k80 output directory
OUTPUT_DIR     = os.getenv("OUTPUT_DIR",   "/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k80_half")

E5_MODEL_NAME  = os.getenv("E5_MODEL_NAME", "intfloat/multilingual-e5-large")

# --- Retrieval sizes ---
K_E5           = int(os.getenv("K_E5", "80"))
RRF_POOL_MULT  = float(os.getenv("RRF_POOL_MULT", "3"))
RRF_POOL_MIN   = int(os.getenv("RRF_POOL_MIN", "150"))
RRF_K          = int(os.getenv("RRF_K", "60"))

# --- Fusion weights ---
WEIGHT_E5            = float(os.getenv("WEIGHT_E5", "0.6"))
WEIGHT_TFIDF         = float(os.getenv("WEIGHT_TFIDF", "0.4"))
WEIGHT_TFIDF_CHAR    = float(os.getenv("WEIGHT_TFIDF_CHAR", "0.15"))

# --- TF-IDF options ---
TFIDF_MAX_FEATS = int(os.getenv("TFIDF_MAX_FEATS", "300000"))
TFIDF_NGRAM_MIN = int(os.getenv("TFIDF_NGRAM_MIN", "1"))
TFIDF_NGRAM_MAX = int(os.getenv("TFIDF_NGRAM_MAX", "2"))

# --- Char TF-IDF ---
ENABLE_CHAR_TFIDF     = int(os.getenv("ENABLE_CHAR_TFIDF", "0"))
TFIDF_CHAR_MIN        = int(os.getenv("TFIDF_CHAR_MIN", "3"))
TFIDF_CHAR_MAX        = int(os.getenv("TFIDF_CHAR_MAX", "4"))
TFIDF_CHAR_MAX_FEATS  = int(os.getenv("TFIDF_CHAR_MAX_FEATS", "200000"))
TFIDF_CHAR_ANALYZER   = os.getenv("TFIDF_CHAR_ANALYZER", "char_wb")

VAL_SIZE       = float(os.getenv("VAL_SIZE", "0.6"))
SEED           = int(os.getenv("SEED", "42"))

SPLIT_VAL             = int(os.getenv("SPLIT_VAL", "1"))
FILTER_VAL_LEAKAGE    = int(os.getenv("FILTER_VAL_LEAKAGE", "1"))

# NEW: reuse previous split (train/val query uuid lists)
# If you set REUSE_SPLIT_DIR, we will look for train_query_uuids.txt and val_query_uuids.txt inside it.
REUSE_SPLIT_DIR = os.getenv("REUSE_SPLIT_DIR", "")
TRAIN_QIDS_SRC  = os.getenv("TRAIN_QIDS_SRC", os.path.join(REUSE_SPLIT_DIR, "train_query_uuids.txt") if REUSE_SPLIT_DIR else "")
VAL_QIDS_SRC    = os.getenv("VAL_QIDS_SRC",   os.path.join(REUSE_SPLIT_DIR, "val_query_uuids.txt")   if REUSE_SPLIT_DIR else "")
STRICT_REUSE    = int(os.getenv("STRICT_REUSE", "0"))  # 1 = raise if any qids missing; 0 = warn and proceed

# Embedding cache
EMB_CACHE_DIR        = os.getenv("EMB_CACHE_DIR", "/content/e5_cache")
USE_MEMMAP_ON_LOAD   = bool(int(os.getenv("USE_MEMMAP_ON_LOAD", "1")))
Path(EMB_CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Where to save stage-1 outputs
STAGE1_DIR = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(STAGE1_DIR).mkdir(parents=True, exist_ok=True)
GROUPS_TRAIN_JSONL = os.getenv("GROUPS_TRAIN_JSONL", os.path.join(STAGE1_DIR, f"groups_train_k{K_E5}.jsonl"))
GROUPS_VAL_JSONL   = os.getenv("GROUPS_VAL_JSONL",   os.path.join(STAGE1_DIR, f"groups_val_k{K_E5}.jsonl"))
META_JSON          = os.getenv("STAGE1_META_JSON",   os.path.join(STAGE1_DIR, "meta.json"))

# NEW: paths for query UUID lists (saved for THIS run)
TRAIN_QIDS_TXT      = os.path.join(STAGE1_DIR, "train_query_uuids.txt")
VAL_QIDS_TXT        = os.path.join(STAGE1_DIR, "val_query_uuids.txt")
QID_SPLIT_JSON      = os.path.join(STAGE1_DIR, "query_uuid_splits.json")


# ---------------------- Setup ----------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[CONFIG] device={device}  K_FINAL={K_E5}  VAL_SIZE={VAL_SIZE}  SEED={SEED}  "
      f"SPLIT_VAL={SPLIT_VAL}  FILTER_VAL_LEAKAGE={FILTER_VAL_LEAKAGE}")
print(f"[PATHS] CORPUS={CORPUS_JSONL}  TRAIN={TRAIN_JSONL}  OUT={OUTPUT_DIR}  CACHE={EMB_CACHE_DIR}  STAGE1={STAGE1_DIR}")
print(f"[FUSION] RRF_POOL_MULT={RRF_POOL_MULT}  RRF_POOL_MIN={RRF_POOL_MIN}  RRF_K={RRF_K}  "
      f"WEIGHTS: E5={WEIGHT_E5} TFIDF(word)={WEIGHT_TFIDF} TFIDF(char)={WEIGHT_TFIDF_CHAR}")
print(f"[TFIDF(word)] max_features={TFIDF_MAX_FEATS} ngram=({TFIDF_NGRAM_MIN},{TFIDF_NGRAM_MAX})")
print(f"[TFIDF(char)] enabled={bool(ENABLE_CHAR_TFIDF)} analyzer={TFIDF_CHAR_ANALYZER} "
      f"ngram=({TFIDF_CHAR_MIN},{TFIDF_CHAR_MAX}) max_features={TFIDF_CHAR_MAX_FEATS}")

# ---------------------- IO ----------------------
def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o = json.loads(line)
            uid = o.get("uuid") or o.get("id")
            if uid: corpus[uid] = o.get("passage") or o.get("text") or ""
    return corpus

def _as_int(x):
    if x is None: return 0
    s=str(x);  return int(s) if s.isdigit() else int(float(s))

def load_train(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o = json.loads(line)
            q   = o.get("query","")
            qid = o.get("query_uuid") or None
            case = o.get("case_name")  # keep this
            paras = o.get("paragraphs",{}) or {}
            labels= o.get("target_actions",{}) or {}
            gt = {}
            for i in range(1000):
                pk, lk = f"paragraph_{i}", f"target_action_{i}"
                if pk not in paras or lk not in labels: break
                puid = paras[pk].get("uuid") or paras[pk].get("id")
                rel  = _as_int(labels[lk])
                if puid: gt[puid] = rel
            rows.append({"query_uuid": qid, "query": q, "gt": gt, "case_name": case})
    return rows

corpus = load_corpus(CORPUS_JSONL)
print(f"[DATA] corpus docs={len(corpus):,}")
train_rows = load_train(TRAIN_JSONL)
print(f"[DATA] queries total={len(train_rows):,}")

# ---------------------- E5 retriever ----------------------
class E5Retriever:
    def __init__(self, name):
        self.device = device
        self.tok = AutoTokenizer.from_pretrained(name, use_fast=True)
        try:
            self.model = AutoModel.from_pretrained(
                name, torch_dtype=(torch.float16 if device=='cuda' else None),
                attn_implementation="sdpa"
            ).to(device)
        except TypeError:
            self.model = AutoModel.from_pretrained(
                name, torch_dtype=(torch.float16 if device=='cuda' else None)
            ).to(device)
        self.model.eval()

    @torch.no_grad()
    def embed(self, texts: List[str], is_query=False, bs=128) -> np.ndarray:
        pref = "query: " if is_query else "passage: "
        out = []
        for i in range(0, len(texts), bs):
            enc = self.tok([pref+t for t in texts[i:i+bs]], padding=True, truncation=True,
                           max_length=512, return_tensors="pt").to(device)
            h = self.model(**enc).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1)
            emb = (h*m).sum(1) / m.sum(1).clamp(min=1)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            out.append(emb.cpu())
        return torch.cat(out,0).numpy()

# ---------------------- Embedding cache helpers ----------------------
def _file_sig(p: Path):
    try:
        st = p.stat()
        return f"{p.name}|{st.st_size}|{int(st.st_mtime)}"
    except Exception:
        return f"{p.name}|0|0"

def _emb_cache_key(corpus_path: str, model_name: str, max_len: int, num_docs_hint: int = 0) -> str:
    p = Path(corpus_path)
    base = f"{_file_sig(p)}|{model_name}|L{max_len}|N{num_docs_hint}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()[:16]

def _emb_cache_paths(key: str):
    base = f"e5_{key}"
    e = os.path.join(EMB_CACHE_DIR, base + "_embeddings.npy")
    i = os.path.join(EMB_CACHE_DIR, base + "_ids.json")
    m = os.path.join(EMB_CACHE_DIR, base + "_meta.json")
    fx = os.path.join(EMB_CACHE_DIR, base + "_index.faiss")
    return e, i, m, fx

def load_embeddings_cache(corpus_path: str, model_name: str, max_len: int, expect_n: int):
    key = _emb_cache_key(corpus_path, model_name, max_len, expect_n)
    e, i, m, fx = _emb_cache_paths(key)
    if not (os.path.exists(e) and os.path.exists(i) and os.path.exists(m)):
        return None
    try:
        meta = json.load(open(m, "r", encoding="utf-8"))
        if meta.get("model_name") != model_name: return None
        ids = json.load(open(i, "r", encoding="utf-8"))
        embs = np.load(e, mmap_mode=("r" if USE_MEMMAP_ON_LOAD else None))
        if embs.shape[0] != len(ids): return None
        print(f"[CACHE] E5 embeddings restored from cache: {e} (shape={embs.shape}, memmap={USE_MEMMAP_ON_LOAD})")
        return {"embeddings": embs, "ids": ids, "meta": meta, "faiss_path": fx, "key": key}
    except Exception as ex:
        print("[CACHE] Failed to load cache:", ex)
        return None

def save_embeddings_cache(corpus_path: str, model_name: str, max_len: int, ids: List[str], embs: np.ndarray):
    key = _emb_cache_key(corpus_path, model_name, max_len, len(ids))
    e, i, m, fx = _emb_cache_paths(key)
    arr = np.asarray(embs, dtype=np.float32, order="C")
    np.save(e, arr)
    json.dump(list(ids), open(i, "w", encoding="utf-8"), ensure_ascii=False)
    meta = {"model_name": model_name, "num_documents": len(ids), "dim": int(arr.shape[1]),
            "corpus_path": str(corpus_path), "max_len": int(512)}
    json.dump(meta, open(m, "w", encoding="utf-8"))
    print(f"[CACHE] Saved embeddings: {e}  (ids: {i}, meta: {m})")
    return {"key": key, "emb_path": e, "ids_path": i, "meta_path": m, "faiss_path": fx}

# ---------------------- FAISS helpers (CPU-only) ----------------------
try:
    import faiss
    FAISS=True
except Exception as e:
    print("[FAISS] unavailable:", e)
    FAISS=False

def build_index(xb: np.ndarray):
    xb = np.asarray(xb, dtype=np.float32, order="C")
    idx = faiss.IndexFlatIP(xb.shape[1])   # CPU index
    idx.add(xb)
    print("[FAISS] Using CPU IndexFlatIP")
    return idx

def save_faiss_cpu_index(index, path: str):
    try:
        faiss.write_index(index, path)  # already CPU
        print(f"[CACHE] Saved FAISS CPU index → {path}")
    except Exception as e:
        print("[CACHE] Could not save FAISS index:", e)

def load_faiss_cpu(path: str):
    try:
        idx = faiss.read_index(path)
        print("[FAISS] Loaded CPU index.")
        return idx
    except Exception as e:
        print("[CACHE] Failed to load FAISS index:", e)
        return None

# ---------------------- Build / Load E5 corpus embeddings & index ----------------------
e5 = E5Retriever(E5_MODEL_NAME)
doc_ids = list(corpus.keys())
doc_texts_map = corpus  # uid -> text

# Try cache
cache = load_embeddings_cache(CORPUS_JSONL, E5_MODEL_NAME, 512, expect_n=len(doc_ids))
if cache is not None:
    cached_ids = cache["ids"]
    if len(cached_ids) == len(doc_ids):
        doc_ids = cached_ids
        doc_texts = [doc_texts_map[i] for i in doc_ids]
        X = cache["embeddings"]
        fx_path = cache["faiss_path"]
        if FAISS and os.path.exists(fx_path):
            index = load_faiss_cpu(fx_path)
        elif FAISS:
            index = build_index(X)
            save_faiss_cpu_index(index, fx_path)
        else:
            index = None
    else:
        print("[CACHE] Cached ids count mismatch; recomputing embeddings.")
        cache = None

if cache is None:
    doc_ids = list(corpus.keys())
    doc_texts = [corpus[d] for d in doc_ids]
    print("[E5] Embedding corpus …")
    X = e5.embed(doc_texts, is_query=False, bs=128)
    print(f"[E5] Corpus embeddings: {X.shape}")
    saved_meta = save_embeddings_cache(CORPUS_JSONL, E5_MODEL_NAME, 512, doc_ids, X)
    if FAISS:
        index = build_index(X)
        save_faiss_cpu_index(index, saved_meta["faiss_path"])
    else:
        index = None

# Fallback vector-matmul for search if FAISS missing
if not FAISS and isinstance(X, np.memmap):
    Xdot = np.array(X).T
elif not FAISS:
    Xdot = X.T

# ---------------------- TF-IDF (word) build ----------------------
print("[TFIDF] Fitting word TF-IDF on corpus …")
tfidf_vec = TfidfVectorizer(max_features=TFIDF_MAX_FEATS,
                            ngram_range=(TFIDF_NGRAM_MIN, TFIDF_NGRAM_MAX))
tfidf_mat = tfidf_vec.fit_transform(doc_texts).astype(np.float32)  # CSR float32
print(f"[TFIDF] Matrix shape={tfidf_mat.shape} nnz={tfidf_mat.nnz:,}")

# ---------------------- TF-IDF (char) build ----------------------
if ENABLE_CHAR_TFIDF:
    print("[TFIDF-CHAR] Fitting char TF-IDF on corpus …")
    tfidf_char_vec = TfidfVectorizer(
        analyzer=TFIDF_CHAR_ANALYZER,
        ngram_range=(TFIDF_CHAR_MIN, TFIDF_CHAR_MAX),
        max_features=TFIDF_CHAR_MAX_FEATS,
        dtype=np.float32,
    )
    tfidf_char_mat = tfidf_char_vec.fit_transform(doc_texts).tocsr().astype(np.float32, copy=False)
    print(f"[TFIDF-CHAR] Matrix shape={tfidf_char_mat.shape} nnz={tfidf_char_mat.nnz:,}")
else:
    tfidf_char_vec, tfidf_char_mat = None, None
    print("[TFIDF-CHAR] Disabled.")

# ---------------------- RRF utilities ----------------------
def rrf_fuse(e5: List[Tuple[int, float]],
             lex: List[Tuple[int, float]],
             k: int,
             lex_c: List[Tuple[int, float]] = None) -> List[int]:
    """
    Fuse up to three ranked lists (indices in doc_ids) using weighted RRF.
    Accepts: e5 (pairs), lex word (pairs), optional lex char (pairs).
    """
    # Prepare rank maps
    sources = []
    weights = []
    if e5:
        sources.append({gi: r for r, (gi, _) in enumerate(e5)})
        weights.append(WEIGHT_E5)
    if lex:
        sources.append({gi: r for r, (gi, _) in enumerate(lex)})
        weights.append(WEIGHT_TFIDF)
    if lex_c:
        sources.append({gi: r for r, (gi, _) in enumerate(lex_c)})
        weights.append(WEIGHT_TFIDF_CHAR)

    if not sources:
        return []

    w_sum = sum(weights) if sum(weights) > 0 else 1.0
    norm_w = [w / w_sum for w in weights]

    universe = set().union(*[set(s.keys()) for s in sources])
    fused = []
    BIG = 10**9
    for gi in universe:
        s = 0.0
        for m, w in zip(sources, norm_w):
            r = m.get(gi, BIG)
            if r < BIG:
                s += float(w) / (RRF_K + r)
        fused.append((gi, s))

    fused.sort(key=lambda x: x[1], reverse=True)
    return [gi for gi, _ in fused[:k]]

# ---------------------- Per-source top-k (with scores) ----------------------
@torch.no_grad()
def e5_topk_with_scores(query: str, k: int):
    qv = e5.embed([query], is_query=True, bs=1).astype(np.float32)
    if FAISS:
        D, I = index.search(qv, min(k, len(doc_ids)))
        pairs = [(int(i), float(d)) for d, i in zip(D[0], I[0]) if i >= 0]
        return pairs  # already sorted by score desc
    else:
        sims = (qv @ Xdot)[0]
        ords = np.argsort(sims)[::-1][:k]
        return [(int(i), float(sims[i])) for i in ords]

def tfidf_topk_with_scores(query: str, k: int) -> List[Tuple[int, float]]:
    qv = tfidf_vec.transform([query]).astype(np.float32)   # 1 x F (CSR float32)
    prod = (tfidf_mat @ qv.T)                              # N x 1 sparse
    scores = (prod.toarray().ravel()
              if hasattr(prod, "toarray") else np.asarray(prod).ravel())
    scores = scores.astype(np.float32, copy=False)
    take = min(k, scores.shape[0])
    if take <= 0:
        return []
    part = np.argpartition(scores, -take)[-take:]
    ords = part[np.argsort(scores[part])[::-1]]
    return [(int(i), float(scores[i])) for i in ords]

def tfidf_char_topk_with_scores(query: str, k: int) -> List[Tuple[int, float]]:
    """Sparse top-k on char TF-IDF (no dense materialization)."""
    if not ENABLE_CHAR_TFIDF or tfidf_char_vec is None or tfidf_char_mat is None or k <= 0:
        return []
    v = tfidf_char_vec.transform([query])      # 1 x F CSR
    s = (tfidf_char_mat @ v.T).tocoo()         # N x 1 sparse
    if s.nnz == 0:
        return []
    take = min(k, s.nnz)
    part = np.argpartition(s.data, -take)[-take:]
    ords = part[np.argsort(s.data[part])[::-1]]
    return [(int(s.row[i]), float(s.data[i])) for i in ords]

def fused_topk_ids(query: str, k_final: int) -> List[str]:
    pool_k = max(int(k_final * RRF_POOL_MULT), RRF_POOL_MIN)
    e5_list        = e5_topk_with_scores(query, pool_k)
    tfidf_list     = tfidf_topk_with_scores(query, pool_k)
    tfidf_char_list= tfidf_char_topk_with_scores(query, pool_k)
    fused_idx  = rrf_fuse(e5_list, tfidf_list, k_final, lex_c=tfidf_char_list)
    return [doc_ids[i] for i in fused_idx]

# ---------------------- Build top-K (fused) groups for every query ----------------------
# Each group: {'query', 'query_uuid', 'case_name', 'pids', 'texts', 'labels'}
groups = []
retrieved_relevant = 0
total_relevant = 0
queries_with_relevant = 0
for row in train_rows:
    pids   = fused_topk_ids(row["query"], K_E5)
    texts  = [doc_texts_map[pid] for pid in pids]
    labels = [int(row["gt"].get(pid, 0)) for pid in pids]  # unlabeled -> 0

    groups.append({
        "query": row["query"],
        "query_uuid": row.get("query_uuid"),
        "case_name": row.get("case_name"),
        "pids": pids,
        "texts": texts,
        "labels": labels
    })

    rel_total = sum(1 for v in row["gt"].values() if v > 0)
    if rel_total:
        queries_with_relevant += 1
        total_relevant += rel_total
        retrieved_relevant += sum(1 for pid in pids if row["gt"].get(pid, 0) > 0)

print(f"[GROUPS] built fused (E5 ⊕ TF-IDF[word]{' ⊕ TF-IDF[char]' if ENABLE_CHAR_TFIDF else ''}) top-{K_E5} for all queries.")
if total_relevant:
    recall_at_k = retrieved_relevant / total_relevant
    print(f"[METRICS] Recall@{K_E5} = {recall_at_k:.4f} ({retrieved_relevant}/{total_relevant}) "
          f"over {queries_with_relevant} queries with relevance.")
else:
    print(f"[METRICS] Recall@{K_E5} undefined (no relevant labels).")

# ---------------------- Upper-bound NDCG@P (perfect reranker) — per corpus ----------------------
from collections import defaultdict

NDCG_P = int(os.getenv("NDCG_P", "20"))

def _dcg_at_p(rels, p):
    s = 0.0
    for i, r in enumerate(rels[:p], start=1):
        s += (2**int(r) - 1) / math.log2(i + 1)
    return s

per_sum = defaultdict(float)
per_cnt = defaultdict(int)
overall_sum, overall_cnt = 0.0, 0

# Use train_rows (full GT) for IDCG; groups (retrieved pool) for DCG upper bound
for row, g in zip(train_rows, groups):
    # Ideal DCG from all known labels for this query
    gt_rels = sorted([int(v) for v in row["gt"].values()], reverse=True)
    idcg = _dcg_at_p(gt_rels, NDCG_P)
    if idcg <= 0:
        continue  # skip queries with no positives

    # Perfect reranker within the retrieved pool (your fused top-K_E5)
    pool_rels = sorted([int(l) for l in g["labels"]], reverse=True)
    ub_ndcg = _dcg_at_p(pool_rels, NDCG_P) / idcg

    tag = g.get("case_name") or "UNKNOWN"
    per_sum[tag] += ub_ndcg
    per_cnt[tag] += 1
    overall_sum += ub_ndcg
    overall_cnt += 1

print(f"[UB][NDCG] Perfect-reranker upper bound NDCG@{NDCG_P} by corpus:")
for tag in sorted(per_sum.keys()):
    mean = per_sum[tag] / per_cnt[tag] if per_cnt[tag] else float("nan")
    print(f"    {tag:12s}: {mean:.4f}  (n={per_cnt[tag]})")
if overall_cnt:
    print(f"[UB][NDCG] Overall: {overall_sum/overall_cnt:.4f}  (n={overall_cnt})")

# ---------------------- Free heavy objects to reduce RAM/VRAM ----------------------
try: del e5
except Exception: pass
try: del index
except Exception: pass
for _name in ("X", "Xdot", "doc_texts", "doc_texts_map", "tfidf_mat", "tfidf_vec",
              "tfidf_char_mat", "tfidf_char_vec"):
    if _name in globals(): globals()[_name] = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("[CLEANUP] Freed E5/FAISS & TF-IDF; cleared CUDA cache.")

# ---------------------- Split by query + optional leakage filter (REUSE if provided) ----------------------
def _read_lines(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return [ln.strip() for ln in f if ln.strip()]

def _filter_train_docs(train_groups, banned):
    kept, drop_items, drop_groups = [], 0, 0
    for g in train_groups:
        mask = [pid not in banned for pid in g["pids"]]
        if not any(mask):
            drop_groups += 1
            continue
        pids  = [p for p,m in zip(g["pids"], mask) if m]
        texts = [t for t,m in zip(g["texts"], mask) if m]
        labs  = [l for l,m in zip(g["labels"],mask) if m]
        drop_items += (len(g["pids"]) - len(pids))
        kept.append({
            "query": g["query"],
            "query_uuid": g.get("query_uuid"),
            "case_name": g.get("case_name"),
            "pids": pids, "texts": texts, "labels": labs
        })
    return kept, drop_items, drop_groups

reuse_available = (TRAIN_QIDS_SRC and os.path.exists(TRAIN_QIDS_SRC)) and \
                  (VAL_QIDS_SRC   and os.path.exists(VAL_QIDS_SRC))

if reuse_available:
    print(f"[SPLIT] Reusing split from:\n"
          f"       train_qids: {TRAIN_QIDS_SRC}\n"
          f"       val_qids:   {VAL_QIDS_SRC}")

    src_train_qids = set(_read_lines(TRAIN_QIDS_SRC))
    src_val_qids   = set(_read_lines(VAL_QIDS_SRC))

    all_qids = {g.get("query_uuid") for g in groups if g.get("query_uuid") is not None}
    missing_train = src_train_qids - all_qids
    missing_val   = src_val_qids   - all_qids

    if missing_train:
        msg = f"[SPLIT][WARN] {len(missing_train)} train qids not present in current groups (first 5): {list(sorted(missing_train))[:5]}"
        print(msg)
        if STRICT_REUSE: raise RuntimeError(msg)
    if missing_val:
        msg = f"[SPLIT][WARN] {len(missing_val)} val qids not present in current groups (first 5): {list(sorted(missing_val))[:5]}"
        print(msg)
        if STRICT_REUSE: raise RuntimeError(msg)

    groups_train = [g for g in groups if g.get("query_uuid") in src_train_qids]
    groups_val   = [g for g in groups if g.get("query_uuid") in src_val_qids]
    print(f"[SPLIT] train groups={len(groups_train)}  val groups={len(groups_val)}  (reused split)")

    if FILTER_VAL_LEAKAGE:
        val_doc_set = set(pid for gv in groups_val for pid in gv["pids"])
        groups_train, n_drop_items, n_drop_groups = _filter_train_docs(groups_train, val_doc_set)
        print(f"[LEAKAGE] removed {n_drop_items} train items; dropped {n_drop_groups} fully-overlapping train groups")
    else:
        print("[LEAKAGE] Skipping leakage removal — train may include docs also in val (submission-style).")

else:
    # Original random split (no reuse files found)
    if SPLIT_VAL:
        idxs = list(range(len(groups)))
        random.shuffle(idxs)
        cut = int(len(groups)*(1.0-VAL_SIZE))
        groups_train = [groups[i] for i in idxs[:cut]]
        groups_val   = [groups[i] for i in idxs[cut:]]
        print(f"[SPLIT] train groups={len(groups_train)}  val groups={len(groups_val)}  (VAL_SIZE={VAL_SIZE})")

        if FILTER_VAL_LEAKAGE:
            val_doc_set = set(pid for gv in groups_val for pid in gv["pids"])
            groups_train, n_drop_items, n_drop_groups = _filter_train_docs(groups_train, val_doc_set)
            print(f"[LEAKAGE] removed {n_drop_items} train items; dropped {n_drop_groups} fully-overlapping train groups")
        else:
            print("[LEAKAGE] Skipping leakage removal — train may include docs also in val (submission-style).")
    else:
        groups_train = groups
        groups_val   = []
        print(f"[SPLIT] no validation split (SPLIT_VAL=0): using ALL {len(groups_train)} groups for training.")

# ---------------------- Save stage-1 outputs ----------------------
def _write_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for obj in items:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def _write_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        for ln in lines:
            f.write(str(ln) + "\n")

_write_jsonl(GROUPS_TRAIN_JSONL, groups_train)
_write_jsonl(GROUPS_VAL_JSONL, groups_val)

# --- save the query UUIDs of train/val splits ---
train_qids = [g.get("query_uuid") for g in groups_train if g.get("query_uuid") is not None]
val_qids   = [g.get("query_uuid") for g in groups_val   if g.get("query_uuid") is not None]

_write_lines(TRAIN_QIDS_TXT, train_qids)
_write_lines(VAL_QIDS_TXT, val_qids)

with open(QID_SPLIT_JSON, "w", encoding="utf-8") as f:
    json.dump({"train_query_uuids": train_qids, "val_query_uuids": val_qids}, f, ensure_ascii=False, indent=2)

meta = {
    # include both keys so downstream code can read either
    "K_FINAL": K_E5,
    "K_E5": K_E5,
    "VAL_SIZE": VAL_SIZE, "SEED": SEED,
    "split_val": int(SPLIT_VAL), "filter_val_leakage": int(FILTER_VAL_LEAKAGE),
    "corpus_path": CORPUS_JSONL, "train_path": TRAIN_JSONL,
    "e5_model": E5_MODEL_NAME, "groups_train": len(groups_train), "groups_val": len(groups_val),
    "fusion": {
        "type": "RRF",
        "RRF_K": RRF_K,
        "RRF_POOL_MULT": RRF_POOL_MULT,
        "RRF_POOL_MIN": RRF_POOL_MIN,
        "weights": {
            "e5": WEIGHT_E5,
            "tfidf_word": WEIGHT_TFIDF,
            "tfidf_char": WEIGHT_TFIDF_CHAR
        }
    },
    "tfidf": {
        "word": {"max_features": TFIDF_MAX_FEATS, "ngram": [TFIDF_NGRAM_MIN, TFIDF_NGRAM_MAX]},
        "char": {
            "enabled": int(ENABLE_CHAR_TFIDF),
            "analyzer": TFIDF_CHAR_ANALYZER,
            "ngram": [TFIDF_CHAR_MIN, TFIDF_CHAR_MAX],
            "max_features": TFIDF_CHAR_MAX_FEATS
        }
    },
    # NEW: include counts for convenience
    "train_query_uuids_path": TRAIN_QIDS_TXT,
    "val_query_uuids_path": VAL_QIDS_TXT,
    "query_uuid_splits_json": QID_SPLIT_JSON,
    "train_query_uuids_count": len(train_qids),
    "val_query_uuids_count": len(val_qids),
}
with open(META_JSON, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"[SAVE] groups_train → {GROUPS_TRAIN_JSONL}")
print(f"[SAVE] groups_val   → {GROUPS_VAL_JSONL}")
print(f"[SAVE] train_query_uuids.txt → {TRAIN_QIDS_TXT}  (n={len(train_qids)})")
print(f"[SAVE] val_query_uuids.txt   → {VAL_QIDS_TXT}    (n={len(val_qids)})")
print(f"[SAVE] query_uuid_splits.json→ {QID_SPLIT_JSON}")
print(f"[SAVE] meta         → {META_JSON}")

## Step 3: Derive Per-Corpus Rating Tables (r_table)

Grid search to find optimal monotone gain mappings `r_y` for labels 0-4 per corpus.

**Objective**: Match target perplexity while maximizing mass on high-relevance labels (3,4).

**Parameters searched**:
- `alpha`: Exponent for base gains (sharper vs softer distributions)
- `delta`: Additive shift before exponentiation
- `beta`: Inverse-frequency weighting strength

**Output**: `ideal_r_no_training.json` with per-corpus r_tables used in ListNet training.

In [2]:
# === One-cell: derive ideal monotone ratings r_y per corpus (no training required) ===
# Uses your existing groups_train/groups_val with fields:
#   - group["labels"]: list[int] of relevance labels in {0,1,2,3,4}
#   - group["case_name"]: e.g., "mafat_retrieval_wikipedia_corpus" / "_kz_corpus" / "_knesset_corpus"
# Produces a JSON with best configs and prints diagnostics.
import numpy as np, json, os, itertools
from dataclasses import dataclass
from typing import Dict, List, Tuple, Iterable

# ---- Sanity: require groups_train/groups_val in scope ----
if 'groups_train' not in globals() or 'groups_val' not in globals():
    raise RuntimeError("Expected 'groups_train' and 'groups_val' to be defined in the notebook scope.")

# ---- Resolve corpus mapping (robust to missing globals) ----
def _infer_slug_map(groups_all: List[dict]) -> Tuple[Dict[str,str], Dict[str,str]]:
    """
    Returns (SLUG_TO_CORPUSKEY, CORPUSKEY_TO_SLUG).
    Prefers existing globals, else infers from case_name values.
    """
    if 'SLUG_TO_CORPUSKEY' in globals() and isinstance(SLUG_TO_CORPUSKEY, dict) and len(SLUG_TO_CORPUSKEY) > 0:
        stc = dict(SLUG_TO_CORPUSKEY)
        cts = {v:k for k,v in stc.items()}
        return stc, cts
    if 'CORPUS_KEYS' in globals() and isinstance(CORPUS_KEYS, dict) and len(CORPUS_KEYS) > 0:
        cts = dict(CORPUS_KEYS)
        stc = {v:k for k,v in cts.items()}
        return stc, cts

    # Infer from case_name strings
    case_names = set([g.get("case_name","") for g in groups_all if g.get("case_name")])
    cts = {}
    for cn in case_names:
        low = cn.lower()
        if "wikipedia" in low:
            cts["wiki"] = cn
        elif "knesset" in low:
            cts["knesset"] = cn
        elif low.endswith("_kz_corpus") or "kol" in low or "zchut" in low or "kz" in low:
            cts["kz"] = cn
    # If anything missing, just map the remaining in deterministic order
    remaining = [cn for cn in case_names if cn not in cts.values()]
    for cn in remaining:
        if "wiki" not in cts: cts["wiki"] = cn; continue
        if "kz" not in cts: cts["kz"] = cn; continue
        if "knesset" not in cts: cts["knesset"] = cn; continue
    stc = {v:k for k,v in cts.items()}
    return stc, cts

_all_groups_for_map = groups_train + groups_val
SLUG_TO_CORPUSKEY_RESOLVED, CORPUSKEY_TO_SLUG_RESOLVED = _infer_slug_map(_all_groups_for_map)

# ---- Helpers over groups ----
def _groups_for_corpus(groups: List[dict], corpus_key: str) -> List[dict]:
    return [g for g in groups if g.get("case_name") == corpus_key]

def _label_counts_per_group(groups: List[dict]) -> List[np.ndarray]:
    out = []
    for g in groups:
        labs = g.get("labels", [])
        cnt = np.zeros(5, dtype=np.int64)
        for l in labs:
            li = int(l)
            if 0 <= li <= 4: cnt[li] += 1
        out.append(cnt)
    return out

def _filter_groups_with_pos(counts_list: List[np.ndarray]) -> List[np.ndarray]:
    return [c for c in counts_list if int(c[1:].sum()) > 0]

def _monotone_nondec(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float64, copy=True)
    for i in range(1, len(arr)):
        if arr[i] < arr[i-1]:
            arr[i] = arr[i-1]
    return arr

# ---- Build weights from label frequency (inverse-frequency^beta), enforce monotone & normalize ----
def _build_weights_from_freq(freq: np.ndarray, beta: float) -> np.ndarray:
    eps = 1e-6
    w = (freq.astype(np.float64) + eps) ** (-float(beta))
    w = _monotone_nondec(w)
    # normalize: w0=0, scale so w1≈1
    w -= w[0]
    denom = max(w[1], 1e-6)
    w = w / denom
    w[0] = 0.0
    return w

# ---- Compute entropy/perplexity and “mass on high labels” for a given r-table ----
_BASE_G = np.array([0.0, 1.0, 3.0, 7.0, 15.0], dtype=np.float64)

@dataclass
class RCfg:
    alpha: float
    delta: float
    beta: float
    weights: np.ndarray  # length 5

    def as_dict(self):
        return {"alpha": float(self.alpha), "delta": float(self.delta),
                "beta": float(self.beta), "w": [float(x) for x in self.weights]}

def _r_table(cfg: RCfg) -> np.ndarray:
    r = (_BASE_G + cfg.delta) ** cfg.alpha
    r *= cfg.weights
    r[0] = 0.0
    # Enforce strictly increasing for labels 1..4
    r = _monotone_nondec(r)
    for i in range(2, 5):  # ensure strictness
        if r[i] <= r[i-1]:
            r[i] = np.nextafter(r[i-1], np.inf)
    # Ensure label-1 has positive mass if any positives exist
    if r[1] <= 0: r[1] = np.finfo(np.float64).tiny
    return r

def _group_metrics_from_r(counts_list: List[np.ndarray], r_tab: np.ndarray) -> Dict[str, float]:
    H_vals, p4_vals, p3_vals = [], [], []
    for c in counts_list:
        # total mass across items (label-0 contributes 0)
        m = float(np.dot(c, r_tab))
        if m <= 0.0:
            continue
        p_lbl = (c * r_tab) / m  # distribution over labels (0..4), sums to 1 over positive labels
        mask = (r_tab > 0) & (c > 0)
        # Entropy across items (labels expanded by counts): H = -sum_y c_y*(r_y/m) * log(r_y/m)
        H = float(-np.sum(p_lbl[mask] * np.log(r_tab[mask])) + np.log(m))
        H_vals.append(H)
        p4_vals.append(float(p_lbl[4]))
        p3_vals.append(float(p_lbl[3]))
    if len(H_vals) == 0:
        return {"avg_H": 0.0, "avg_perplex": 0.0, "p4": 0.0, "p3": 0.0}
    avg_H = float(np.mean(H_vals))
    return {"avg_H": avg_H, "avg_perplex": float(np.exp(avg_H)),
            "p4": float(np.mean(p4_vals)), "p3": float(np.mean(p3_vals))}

# ---- Target perplexity chooser (data-driven, no training) ----
def _choose_target_perplexity(counts_list: List[np.ndarray], gamma: float,
                              lo: float = 2.0, hi: float = 14.0) -> float:
    pos_counts = [int(c[1:].sum()) for c in counts_list if int(c[1:].sum()) > 0]
    if not pos_counts:
        return lo
    med_pos = float(np.median(pos_counts))
    return float(np.clip(gamma * med_pos, lo, hi))

# ---- Candidate grid per corpus ----
def _grid_for_slug(slug: str) -> Tuple[Iterable[float], Iterable[float], Iterable[float], float, Dict[str, float]]:
    slug = slug.lower()
    if slug == "knesset":
        alphas = np.arange(1.20, 1.71, 0.05)  # sharper
        deltas = [0.00, 0.05, 0.10]
        betas  = [0.25, 0.50, 0.75]
        gamma  = 0.70
        weights = {"wP": 1.00, "w4": 0.55, "w3": 0.25}
    elif slug == "kz":
        alphas = np.arange(0.70, 1.01, 0.05)  # softer
        deltas = [0.10, 0.20, 0.30]
        betas  = [0.00, 0.25, 0.50]
        gamma  = 0.60
        weights = {"wP": 1.00, "w4": 0.35, "w3": 0.20}
    else:  # wiki
        alphas = np.arange(0.90, 1.21, 0.05)
        deltas = [0.00, 0.05, 0.10]
        betas  = [0.00, 0.25, 0.50]
        gamma  = 0.75
        weights = {"wP": 1.00, "w4": 0.45, "w3": 0.20}
    return alphas, deltas, betas, gamma, weights

# ---- Objective: match target perplexity & reward mass on high labels ----
def _objective(metrics: Dict[str, float], target_perplex: float, w: Dict[str, float]) -> float:
    P = metrics["avg_perplex"]
    p4, p3 = metrics["p4"], metrics["p3"]
    return float(w["wP"] * (P - target_perplex) ** 2 - w["w4"] * p4 - w["w3"] * p3)  # lower is better

# ---- Main entry: find r per corpus (no training) ----
def find_ratings_without_training(slugs=("wiki","kz","knesset"),
                                  use_val: bool = True,
                                  save_json_path: str = None) -> Dict[str, dict]:
    results = {}
    for s in slugs:
        slug = s.lower()
        if slug not in SLUG_TO_CORPUSKEY_RESOLVED:
            print(f"[WARN] unknown slug '{s}', available: {list(SLUG_TO_CORPUSKEY_RESOLVED.keys())}")
            continue
        corpus_key = SLUG_TO_CORPUSKEY_RESOLVED[slug]
        tr = _groups_for_corpus(groups_train, corpus_key)
        va = _groups_for_corpus(groups_val,   corpus_key)
        source_groups = tr + (va if use_val else [])
        counts_list = _filter_groups_with_pos(_label_counts_per_group(source_groups))
        if not counts_list:
            print(f"[WARN:{slug}] no groups with positives")
            results[slug] = None
            continue

        # global label freq for weights
        freq = np.sum(np.stack(counts_list, axis=0), axis=0)  # length 5
        alphas, deltas, betas, gamma, w_obj = _grid_for_slug(slug)
        target_perplex = _choose_target_perplexity(counts_list, gamma=gamma)

        best = {"loss": np.inf, "cfg": None, "metrics": None, "target_perplex": target_perplex}

        for a, d, b in itertools.product(alphas, deltas, betas):
            w = _build_weights_from_freq(freq, beta=b)
            cfg = RCfg(alpha=float(a), delta=float(d), beta=float(b), weights=w)
            rtab = _r_table(cfg)
            metrics = _group_metrics_from_r(counts_list, rtab)
            loss = _objective(metrics, target_perplex, w_obj)
            if loss < best["loss"] - 1e-12:
                best = {"loss": loss, "cfg": cfg, "metrics": metrics, "target_perplex": target_perplex}

        cfgd = best["cfg"].as_dict()
        diag = {
            "target_perplex": best["target_perplex"],
            "avg_perplex": best["metrics"]["avg_perplex"],
            "avg_entropy": best["metrics"]["avg_H"],
            "p4_mass": best["metrics"]["p4"],
            "p3_mass": best["metrics"]["p3"],
            "loss": best["loss"]
        }
        results[slug] = {
            "gain_cfg": cfgd,
            "r_table": list(np.round(_r_table(best["cfg"]), 8)),
            "diagnostics": diag,
            "label_freq": [int(x) for x in freq.tolist()]
        }
        print(f"[BEST:{slug}] r={cfgd} | r_table={results[slug]['r_table']} | diag={diag}")

    if save_json_path:
        os.makedirs(os.path.dirname(save_json_path), exist_ok=True)
        with open(save_json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"[SAVE] wrote configs → {save_json_path}")
    return results

# ---- Run it and save JSON next to your OUTPUT_DIR if available ----
_default_out_dir = globals().get("OUTPUT_DIR", ".")
_out_json = os.path.join(_default_out_dir, "ideal_r_no_training.json")
best_ratings = find_ratings_without_training(
    slugs=tuple(SLUG_TO_CORPUSKEY_RESOLVED.keys()),  # ('wiki','kz','knesset') if present
    use_val=True,
    save_json_path=_out_json
)

# The dict 'best_ratings' now contains, per corpus:
#   - "gain_cfg": {alpha, delta, beta, w: [w0..w4]}
#   - "r_table": the resulting monotone ratings [r0..r4] (rounded for readability)
#   - "diagnostics": target/achieved perplexity, entropy, p3/p4 mass and objective
#   - "label_freq": aggregate label counts used for weighting
best_ratings


[BEST:wiki] r={'alpha': 1.2000000000000002, 'delta': 0.1, 'beta': 0.25, 'w': [0.0, 1.0, 1.3388986929922178, 1.4506716596187352, 1.4506716596187352]} | r_table=[np.float64(0.0), np.float64(1.12116936), np.float64(5.2045328), np.float64(15.2433052), np.float64(37.70000983)] | diag={'target_perplex': 3.0, 'avg_perplex': 2.9616531803667923, 'avg_entropy': 1.0857476193097215, 'p4_mass': 0.6893215882500257, 'p3_mass': 0.126244387285568, 'loss': -0.33397311359364334}
[BEST:kz] r={'alpha': 0.9500000000000002, 'delta': 0.3, 'beta': 0.5, 'w': [0.0, 1.0, 2.0134507727473343, 3.189912460588489, 3.189912460588489]} | r_table=[np.float64(0.0), np.float64(1.28305769), np.float64(6.25935032), np.float64(21.0831492), np.float64(42.58293821)] | diag={'target_perplex': 4.2, 'avg_perplex': 4.192157285990464, 'avg_entropy': 1.4332154668278565, 'p4_mass': 0.27635058958992575, 'p3_mass': 0.18823061661737053, 'loss': -0.13430732151691274}
[BEST:knesset] r={'alpha': 1.2, 'delta': 0.1, 'beta': 0.25, 'w': [0.0, 1

{'wiki': {'gain_cfg': {'alpha': 1.2000000000000002,
   'delta': 0.1,
   'beta': 0.25,
   'w': [0.0,
    1.0,
    1.3388986929922178,
    1.4506716596187352,
    1.4506716596187352]},
  'r_table': [np.float64(0.0),
   np.float64(1.12116936),
   np.float64(5.2045328),
   np.float64(15.2433052),
   np.float64(37.70000983)],
  'diagnostics': {'target_perplex': 3.0,
   'avg_perplex': 2.9616531803667923,
   'avg_entropy': 1.0857476193097215,
   'p4_mass': 0.6893215882500257,
   'p3_mass': 0.126244387285568,
   'loss': -0.33397311359364334},
  'label_freq': [55231, 1311, 620, 498, 1540]},
 'kz': {'gain_cfg': {'alpha': 0.9500000000000002,
   'delta': 0.3,
   'beta': 0.5,
   'w': [0.0, 1.0, 2.0134507727473343, 3.189912460588489, 3.189912460588489]},
  'r_table': [np.float64(0.0),
   np.float64(1.28305769),
   np.float64(6.25935032),
   np.float64(21.0831492),
   np.float64(42.58293821)],
  'diagnostics': {'target_perplex': 4.2,
   'avg_perplex': 4.192157285990464,
   'avg_entropy': 1.4332154668